# Synthetic Bootstrapping: Stripping PII

When you have real customer data but cannot use it directly because of names, email addresses, or account numbers, you can ask an LM to generate structurally similar examples with fake details. This notebook walks through one ticket end-to-end, then loops over a batch.

**Required env vars:** `OPENAI_API_KEY` (loaded from `.env`).

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## A real example with sensitive fields

The ticket below has a name, email address, and account number that we don't want to ship into a training set.

In [ ]:
real_example = dspy.Example(
    ticket_text="Hi, this is John Smith (john.smith@acmecorp.com). I need help with account #AC-84932.",
    department="account",
).with_inputs("ticket_text")

# Sensitive: name, email, account number
print(real_example.ticket_text)

## A signature for generating synthetic tickets

The model is asked to preserve structure (greeting, name, email, account number, request) but swap in fake details.

In [ ]:
class SynthesizeTicket(dspy.Signature):
    """Generate a synthetic support ticket with similar structure but fake details."""
    original_ticket: str = dspy.InputField()
    synthetic_ticket: str = dspy.OutputField(
        desc="Synthetic ticket with fake names, emails, and account numbers"
    )

## Generate a single synthetic example

In [ ]:
synthesizer = dspy.ChainOfThought(SynthesizeTicket)

synthetic_result = synthesizer(original_ticket=real_example.ticket_text)
print(synthetic_result.synthetic_ticket)

## Generate a synthetic training set

Loop over a list of real examples and keep the original label but replace the input text with the synthetic version. Replace `real_data` with your own examples.

In [ ]:
# Replace this with your own real-data examples
real_data = [
    real_example,
    dspy.Example(
        ticket_text="Hello, Maria Lopez here (maria.lopez@globex.io). My invoice for account #GX-55120 looks wrong.",
        department="billing",
    ).with_inputs("ticket_text"),
]

synthetic_train_set = []
for example in real_data:
    result = synthesizer(original_ticket=example.ticket_text)
    synthetic_train_set.append(dspy.Example(
        ticket_text=result.synthetic_ticket,
        department=example.department,  # Keep the same label
    ).with_inputs("ticket_text"))

for ex in synthetic_train_set:
    print(ex.department, '->', ex.ticket_text)